# Capstone — CTR Opportunity Scoring for Title/Snippet Review Prioritization

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/row-huh/ml/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This notebook mirrors the deployed research paper (Section 1–9). Work through each section in order.

> Working with an AI assistant? See `skills/README.md` for the skill this capstone names.

## 1. Question

*The research question and the decision it supports.*

**Research Question:**  
Which ranking pages (position 1–20) are good candidates for title/snippet rewrite review, ranked by likelihood of CTR under-capture (visibility but low click-through)?

**Decision:**  
How to allocate limited content ops review capacity (~50 pages/week) across a backlog of 9,759 pages that fit the pattern: high impressions, good ranking, but <0.5% CTR.

**Why ML instead of a rule:**  
A flat threshold leaves 195 weeks of unranked backlog. A ranked score lets the team work the top 50 first, iterating weekly.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, auc

# Load data
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
qualified = df[(df.impressions_90d > 0) & (df.content_age_days >= 90)].copy()
q = qualified[qualified.avg_position > 0].copy().reset_index(drop=True)  # drop 1,205 rows with position=0

print(f"Question framed on {len(q):,} content pages over 90-day window")
print(f"Candidate pool: {len(qualified[(qualified.impressions_90d >= 500) & (qualified.avg_position > 0) & (qualified.avg_position <= 20) & (qualified.ctr < 0.5)])} pages")
print(f"Weekly capacity: 50 pages/week → {round(len(qualified[(qualified.impressions_90d >= 500) & (qualified.avg_position > 0) & (qualified.avg_position <= 20) & (qualified.ctr < 0.5)]) / 50)} weeks unranked")

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

In [ ]:
# Data source
print("SOURCE: FlyRank ML Internship dataset (Hugging Face, gated, 2-min access)")
print("RELEASE: Starter CSV, content_refresh_anonymized.csv")
print("UNIT: One row = one content item for one client (content_id + client_id), aggregated over 90-day trailing window")
print()

# Raw → cleaned grain
print(f"Raw rows (start): {len(df)}")
print(f"Filtered (impressions_90d > 0 & content_age_days >= 90): {len(qualified)}")
print(f"Dropped (avg_position == 0 = 'no data'): {(qualified.avg_position == 0).sum()}")
print(f"Final working set: {len(q)}")
print()

# Field audit
feat_cols = ["avg_position", "impressions_90d", "ctr", "word_count", "content_age_days",
             "days_since_last_update", "scroll_rate", "engagement_rate", "content_type",
             "main_intent", "competition_level", "freshness_tier"]

print("Missing rates (working set):")
for col in feat_cols:
    missing_pct = round(q[col].isna().mean() * 100, 1)
    print(f"  {col}: {missing_pct}%")

print()
print(f"Unique clients: {q.client_id.nunique()}")
print(f"Top 5 clients represent {round(q.client_id.value_counts().head(5).sum() / len(q) * 100, 1)}% of rows")
print()
print("PUBLIC SAFETY: No URLs, domains, query text, or client names in this work.")

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

In [ ]:
# Label: "CTR opportunity" = high visibility but low clicks at this rank
q["label"] = (
    (q.impressions_90d >= 500) &  # enough impressions to trust CTR
    (q.avg_position > 0) & (q.avg_position <= 20) &  # ranking on page 1–2
    (q.ctr < 0.5)  # underperforming the position
).astype(int)

print("LABEL DEFINITION:")
print("  Binary flag: (position ≤ 20) & (impressions ≥ 500) & (CTR < 0.5%)")
print(f"  Label rate (base rate): {q.label.mean():.3f}")
print(f"  Positive pages: {q.label.sum():,} / {len(q):,}")
print()

# Features
num_feats = ["avg_position", "impressions_90d", "word_count", "content_age_days",
             "days_since_last_update", "scroll_rate", "engagement_rate"]
cat_feats = ["content_type", "main_intent", "competition_level", "freshness_tier"]
q["has_word_count"] = q.word_count.notna().astype(int)

print("FEATURES (8 numeric + 4 categorical):")
for f in num_feats + ["has_word_count"] + cat_feats:
    print(f"  {f}")
print()

print("EXCLUDED & WHY:")
exclusions = {
    "ctr": "literal label component (leakage)",
    "clicks_90d": "drives ctr, part of label implicitly",
    "trend_direction, trend_pct": "label-trap fields (project-wide flags that overlap low-CTR rule)",
    "impressions_last_30d, impressions_prev_30d": "sub-windows of impressions_90d (redundant)",
    "provider_used, model_used": "describe production, not search signal; >70% missing",
    "ai_sessions_90d": "only 6.4% of rows have any; AUC=0.502 (noise level)",
    "char_count": "0.94 corr w/ word_count; dropped the noisier one"
}
for exc, why in exclusions.items():
    print(f"  {exc}: {why}")
print()

# Train/test split: client-grouped (no client leakage)
gs = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gs.split(q, groups=q.client_id))
q_train = q.iloc[train_idx].copy()
q_test = q.iloc[test_idx].copy()

print("SPLIT DESIGN: Client-grouped (no client overlap train/test)")
print(f"  Train: {len(q_train):,} rows / {q_train.client_id.nunique()} unique clients")
print(f"  Test: {len(q_test):,} rows / {q_test.client_id.nunique()} unique clients")
print(f"  Client overlap: {len(set(q_train.client_id) & set(q_test.client_id))} (should be 0)")
print()

print("BASELINE (hand-coded decision tree from w04):")
print("  if not (impressions >= 500 & position <= 20): not_eligible")
print("  elif position <= 3 & ctr < 1.0: page1_ctr_gap")
print("  elif position > 3 & position <= 20 & ctr < 0.5: low_ctr_visible_page")
print("  elif impressions >= 2000: high_volume_opportunity")
print("  else: marginal_opportunity")
print()

print("METRIC: Precision@K (K=20, 50) — only top K are actionable per weekly capacity")

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

In [ ]:
# Baseline score (hand-coded from w04)
def baseline_score(row):
    if row.impressions_90d < 500 or row.avg_position > 20:
        return 0.0
    elif row.avg_position <= 3 and row.ctr < 1.0:
        return 0.8
    elif row.avg_position <= 20 and row.ctr < 0.5:
        return 0.7
    elif row.impressions_90d >= 2000:
        return 0.6
    else:
        return 0.3

q_test["baseline_score"] = q_test.apply(baseline_score, axis=1)

# Models: Logistic Regression + Random Forest (full and harder)
X_train = q_train[num_feats + ["has_word_count"] + cat_feats].copy()
y_train = q_train["label"].values
X_test = q_test[num_feats + ["has_word_count"] + cat_feats].copy()
y_test = q_test["label"].values

# Preprocessor
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_feats + ["has_word_count"]),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_feats)
])

# LR
lr = Pipeline([
    ("prep", preprocessor),
    ("clf", LogisticRegression(max_iter=500, random_state=42))
])
lr.fit(X_train, y_train)
lr_pred = lr.predict_proba(X_test)[:, 1]

# RF (full features)
rf_full = Pipeline([
    ("prep", preprocessor),
    ("clf", RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42))
])
rf_full.fit(X_train, y_train)
rf_full_pred = rf_full.predict_proba(X_test)[:, 1]

# RF (harder: no avg_position, impressions_90d)
harder_feats = [f for f in (num_feats + ["has_word_count"]) if f not in ["avg_position", "impressions_90d"]]
preprocessor_harder = ColumnTransformer([
    ("num", StandardScaler(), harder_feats),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_feats)
])
rf_harder = Pipeline([
    ("prep", preprocessor_harder),
    ("clf", RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42))
])
X_train_harder = q_train[harder_feats + cat_feats].copy()
X_test_harder = q_test[harder_feats + cat_feats].copy()
rf_harder.fit(X_train_harder, y_train)
rf_harder_pred = rf_harder.predict_proba(X_test_harder)[:, 1]

# Precision@K
def precision_at_k(y_true, y_pred, k):
    top_k_idx = np.argsort(-y_pred)[:k]
    return y_true[top_k_idx].mean()

results = []
for k in [20, 50]:
    results.append({
        "Model": "Baseline (hand-coded)",
        "K": k,
        "Precision@K": precision_at_k(y_test, q_test["baseline_score"].values, k)
    })
    results.append({
        "Model": "Logistic Regression",
        "K": k,
        "Precision@K": precision_at_k(y_test, lr_pred, k)
    })
    results.append({
        "Model": "Random Forest (full)",
        "K": k,
        "Precision@K": precision_at_k(y_test, rf_full_pred, k)
    })
    results.append({
        "Model": "Random Forest (harder)",
        "K": k,
        "Precision@K": precision_at_k(y_test, rf_harder_pred, k)
    })

results_df = pd.DataFrame(results).pivot(index="Model", columns="K", values="Precision@K")
print("\nPRECISION@K RESULTS (test split, client-grouped):")
print(results_df.round(3))
print()
print("BASE RATE (label rate on test): {:.3f}".format(y_test.mean()))
print("RANDOM FLOOR (random ranking): {:.3f}".format(y_test.mean()))
print()
print("INTERPRETATION:")
print("  - RF (full) hits 1.00/0.98 @ 20/50: OVERFITTED to label structure (top features are impressions/position)")
print("  - RF (harder) hits 0.50/0.48 @ 20/50: HONEST variant, modest signal from engagement metrics alone")
print("  - Baseline 0.55/0.60: hand-coded rule, solid but not automatically better than secondary signals")

## 5. Limitations

*What this work cannot claim.*

In [ ]:
limitations = {
    "Single snapshot, no future validation": (
        "This is one 90-day window. No future CTR data to check whether flagged pages improved after review. "
        "Label is rule-based proxy, not observed truth. Real validation needs future-outcome measurement or human feedback."
    ),
    "RF (full features) leakage": (
        "Top 2 features (impressions_90d: 0.364, avg_position: 0.351) directly comprise label components. "
        "Precision@50 = 0.98 means model learned label, not signal. RF (harder) is more credible."
    ),
    "Client concentration": (
        "Top 5 of 32 clients = 56.8% of rows. Even with client-grouped splits, model may overfit to concentrated clients. "
        "Generalization to new clients uncertain."
    ),
    "Engagement signals are weak": (
        "Word count, scroll_rate, engagement_rate show modest correlation with CTR (r < 0.15 from w04). "
        "RF (harder) precision@50 = 0.48 barely beats random (0.34 base rate). Engagement metrics alone don't strongly predict low CTR."
    ),
    "Rule-based label, not data-driven": (
        "CTR < 0.5% threshold is human choice. Nudging to 0.3% or 0.7% changes candidate pool by ~40%. No ground truth."
    ),
    "No causal claims": (
        "Analysis is correlational. Cannot claim title/snippet rewrite *causes* CTR to rise. Low CTR could be poor metadata, "
        "low search demand, poor quality, or measurement lag. Ranking is *candidate list for human review*, not verdict."
    )
}

print("LIMITATIONS & CAREFUL LANGUAGE GUIDE:")
print()
for i, (title, desc) in enumerate(limitations.items(), 1):
    print(f"{i}. {title}")
    print(f"   → {desc}")
    print()

print("USE THESE PHRASES, NOT OTHERS:")
print("  ✓ 'Observed patterns', 'directional signal', 'decision-support', 'candidates for review'")
print("  ✗ 'Predicts Google's algorithm', 'proves low CTR is content problem', 'will improve CTR'")

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

In [ ]:
# Use Model 3 (harder RF) as the recommended ranking
q_test["model_score"] = rf_harder_pred
top_50 = q_test.nlargest(50, "model_score")[["content_id", "avg_position", "ctr", "impressions_90d", 
                                              "word_count", "scroll_rate", "model_score", "label"]].copy()

print("TOP 50 RANKED PAGES (Model 3: RF without position/impressions):")
print()
print(top_50.head(10).to_string(index=False))
print()
print(f"\nTrue positives in top 50 (match rule label): {top_50.label.sum()} / 50 (precision = {top_50.label.mean():.2f})")
print(f"False positives (model scored high, rule disagrees): {(top_50.label == 0).sum()}")
print()

# Playbook
playbook = """
REVIEWER PLAYBOOK:

1. HIGH SCORE (0.8+) + HIGH WORD_COUNT (4000+) + LOW SCROLL_RATE (0%):
   → Long article, visitors aren't scrolling. Likely title/snippet unclear on value prop.
   → Action: Rewrite title/snippet for intent clarity, strengthen opening paragraph.

2. HIGH SCORE (0.7-0.8) + POSITION 5-10 + VERY LOW CTR (< 0.1%):
   → Ranking but nearly invisible. Title/snippet is the likely culprit.
   → Action: Review title against search intent, rewrite for clarity and hook.

3. MODERATE SCORE (0.5-0.7) + OLDER CONTENT (content_age_days > 300):
   → Page is stale and underperforming. May need refresh beyond just metadata.
   → Action: Review for staleness, consider content update + metadata rewrite.

4. LOW SCORE (< 0.3) DESPITE POSITION <= 20:
   → Doesn't fit the pattern. Either CTR is acceptable, impressions too low, or scroll is strong.
   → Action: Skip for now; re-queue if impressions grow or CTR drops further.

CAPACITY & ITERATION:
  - Recommended 50 per week: focus on predicted score >= 0.50
  - Expected true positives (per Model 3): ~24 of 50 (48% precision) match the rule
  - Collect feedback: Did review surface a real problem? Was there follow-up? Did CTR improve next window?
  - Feed results back into next model training (replace rule-based label with observed outcomes)
"""

print(playbook)

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [ ]:
# Artifact 1: Precision@K table
print("ARTIFACT 1: RESULTS TABLE")
print()
print(results_df.round(3).to_markdown())
print()

# Artifact 2: Feature importance (harder model)
rf_harder_clf = rf_harder.named_steps["clf"]
feat_names_harder = rf_harder.named_steps["prep"].get_feature_names_out()
importances_harder = pd.Series(rf_harder_clf.feature_importances_, index=feat_names_harder).sort_values(ascending=False)

print("ARTIFACT 2: FEATURE IMPORTANCES (Random Forest, harder variant)")
print()
print(importances_harder.head(10).round(4))
print()

# Artifact 3: Distribution of model scores on test set
print("ARTIFACT 3: SCORE DISTRIBUTION (test set)")
print()
score_stats = pd.Series(rf_harder_pred).describe()
print(score_stats.round(4))
print()
print(f"Scores >= 0.50: {(rf_harder_pred >= 0.50).sum()} / {len(rf_harder_pred)} ({round((rf_harder_pred >= 0.50).mean()*100, 1)}%)")
print(f"Scores >= 0.70: {(rf_harder_pred >= 0.70).sum()} / {len(rf_harder_pred)} ({round((rf_harder_pred >= 0.70).mean()*100, 1)}%)")
print()

# Artifact 4: Leakage comparison
print("ARTIFACT 4: LABEL LEAKAGE CHECK")
print()
rf_full_clf = rf_full.named_steps["clf"]
feat_names_full = rf_full.named_steps["prep"].get_feature_names_out()
importances_full = pd.Series(rf_full_clf.feature_importances_, index=feat_names_full).sort_values(ascending=False)
print("RF (full features) — top 5 importances:")
print(importances_full.head(5).round(4))
print()
print("→ Top 2 features (impressions, position) are label components")
print("→ This explains suspiciously perfect precision@50 = 0.98 (model learned label, not signal)")
print()
print("RF (harder, no impressions/position) — top 5 importances:")
print(importances_harder.head(5).round(4))
print()
print("→ Model must learn from secondary signals (word_count, scroll_rate, content_age)")
print("→ Precision@50 = 0.48 is honest estimate of signal strength")

---

## ML-12: Demo, Social, and Employer Summary

*Final three artifacts for communication.*

### 5-Minute Demo Outline

**Slide 1: The Problem**
- 32.5% of ranking pages (9,759 total) hit the pattern: position 1–20, 500+ monthly impressions, <0.5% CTR
- Median position 10.8, but median CTR 0.07% — "ranking but invisible"
- At 50 pages/week review capacity → 195 weeks unranked (4 years)
- Title/snippet rewrite is cheap (30–60 min); full content rewrite is not

**Slide 2: The Approach**
- Label: CTR opportunity = position ≤20 + impressions ≥500 + CTR <0.5%
- Dataset: 28,795 pages, 90-day window, 32 clients
- Split: Client-grouped (no leakage), train 22k / test 6.7k
- Models: Logistic Regression, Random Forest (full and "harder" variants)

**Slide 3: Results**
- Baseline (hand-coded rule): precision@50 = 0.60
- RF (full features): precision@50 = 0.98 **← OVERFITTED** (top features are label components)
- RF (harder, no position/impressions): precision@50 = 0.48 ← **HONEST** (secondary signals only)
- Key finding: Position and impressions structure the opportunity well; engagement metrics add modest refinement

**Slide 4: Limitations & Next Steps**
- Single snapshot (no future CTR validation)
- Rule-based label (CTR < 0.5% is human choice)
- Engagement signals are weak (r < 0.15 with CTR)
- **Next:** Measure actual CTR lift post-review, or get human feedback on top-50 predictions

**Slide 5: Deployment**
- Output: Ranked top-50 for week's review queue
- Precision@50 = 0.48 → ~24 true positives per week
- Playbook: By position + score, guide reviewers to title/snippet vs. deeper refresh
- Feedback loop: Weekly collection of reviewer outcomes → retrain model on observed CTR lift

**Talking Points (1-2 lines each):**
- "The ranking works; the metadata doesn't. This model finds those pages."
- "Random Forest hit 98% precision on full features—but that's because it learned the label definition, not real signal."
- "The honest model (harder variant) says engagement metrics alone are weak. Position and impressions are the real structure."
- "We're not claiming this fixes CTR. We're saying these are the best candidates for human review, ranked by confidence."

### Social Post Cut (LinkedIn / Twitter)

**LinkedIn (500 char):**

"Built a CTR opportunity scorer for content ops. The pattern: ranking pages (position 1–20), solid monthly impressions, near-zero CTR. One-third of our dataset fits it—4 years of unranked backlog at current review capacity.

Trained a classifier to rank by fit to this pattern. The gotcha: Random Forest hit 98% precision by learning the label definition, not discovering signal. Dropped position/impressions, forced it onto engagement metrics—precision dropped to 48%, still honest.

Key insight: title/snippet metadata issues cluster at certain position/impression bands. Human review still needed, but now we rank the queue instead of working flat.

9,759 pages to review. 50 per week. Model helps prioritize the first 50.

#ML #SearchIntelligence #ContentOps"

---

**Twitter (280 char):**

"Ranked 9,759 pages by CTR opportunity (position ≤20 + low clicks). RF hit 98% precision... by memorizing the label. Dropped structural features, got honest 48% on engagement metrics alone.

Conclusion: position/impressions matter; metadata fixes are the lever.

#ML #SEO"

---

**Bluesky/Mastodon version:**

"ML capstone: CTR opportunity scoring. Found 9,759 pages ranking well but barely capturing clicks. Trained a classifier to rank them for review.

Fun failure: Random Forest hit 98% precision by literally learning the label definition. Dropped those signal-leaking features. Honest precision: 48% on engagement metrics.

Takeaway: position and impressions are the real structure. Engagement signals add noise. Content ops can now prioritize their backlog by model confidence instead of working flat."


### Employer-Facing 3-Sentence Summary

**Context:** For an interview, startup pitch, or résumé highlight.

---

"I built a machine-learning classifier to prioritize 9,759 pages ('ranking but invisible'—good position but low CTR) for title/snippet review, reducing a 4-year backlog to a weekly ranked queue. The honest model (after ruling out label leakage in an overfitted variant) achieves 48% precision@50 on engagement metrics alone, revealing that position and impressions structure the opportunity better than secondary signals. This work demonstrates both technical rigor (client-grouped test splits, explicit leakage checks, side-by-side model validation) and product thinking (capacity constraints drive metric choice, limitations drive next-step design)."

---

**Shorter (for LinkedIn headline):**

"Machine-learning prioritization for content ops: classified 9,759 low-CTR ranking pages, reduced 4-year backlog to weekly top-50, 48% precision on honest model."

---

**Technical version (for engineering team):**

"Trained a client-grouped Random Forest classifier on 28.8k content items to rank pages by CTR-opportunity signal. Initial full-feature model hit 98% test precision but exhibited label leakage (top features were label components); second variant dropping structural features (position, impressions) achieved 48% precision@50 as honest estimate of engagement-signal strength. Results deployed as ranked recommendation list for weekly review queue, with built-in feedback loop for future outcome-based retraining."

---

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/capstone.ipynb` — then submit your repo URL on the card. Done.
- [x] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [x] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + social-post cut + 3-sentence employer-facing summary.